In [0]:
from pyspark.sql.functions import col, sum

# ------------------------------
# Step 1 : Read Raw CSV (Bronze Input)
# ------------------------------

df = spark.read.csv(
"/Volumes/workspace/default/my_volume_1/medallion/medallion_data.csv",
header=True,
inferSchema=True
)

print("Raw Data")
display(df)


# ------------------------------
# Step 2 : Bronze Layer
# ------------------------------

df.write \
.mode("overwrite") \
.option("header","true") \
.csv("/Volumes/workspace/default/my_volume_1/medallion/bronze_csv")

print("Bronze Layer Created")


# ------------------------------
# Step 3 : Silver Layer (Clean Data)
# ------------------------------

df_silver = df.filter(col("price") > 0)

display(df_silver)

df_silver.write \
.mode("overwrite") \
.option("header","true") \
.csv("/Volumes/workspace/default/my_volume_1/medallion/silver_csv")

print("Silver Layer Created")


# ------------------------------
# Step 4 : Gold Layer (Aggregation)
# ------------------------------

df_gold = df_silver.groupBy("product") \
.agg(sum("price").alias("total_sales"))

display(df_gold)

df_gold.write \
.mode("overwrite") \
.option("header","true") \
.csv("/Volumes/workspace/default/my_volume_1/medallion/gold_csv")

print("Gold Layer Created")